<a href="https://colab.research.google.com/github/reneemanzari-ship-it/stock-market-news-sentiment-analysis/blob/main/04_Medical_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

1. **Critical Care Protocols:** "What is the protocol for managing sepsis in a critical care unit?"

2. **General Surgery:** "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

3. **Dermatology:** "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

4. **Neurology:** "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"


### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## **Please read the instructions carefully before starting the project.**

This is a commented Python Notebook file in which all the instructions and tasks to be performed are mentioned.
* Blanks '_____' are provided in the notebook that
needs to be filled with an appropriate code to get the correct result. With every '_____' blank, there is a comment that briefly describes what needs to be filled in the blank space.
* Identify the task to be performed correctly, and only then proceed to write the required code.
* Please run the codes in a sequential manner from the beginning to avoid any unnecessary errors.
* Add the results/observations (wherever mentioned) derived from the analysis in the presentation and submit the same. Any mathematical or computational details which are a graded part of the project can be included in the Appendix section of the presentation.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# Install required libraries
!pip install -q langchain_community==0.3.27 \
              langchain==0.3.27 \
              chromadb==1.0.15 \
              pymupdf==1.26.3 \
              tiktoken==0.9.0 \
              datasets==4.0.0 \
              evaluate==0.4.5 \
              langchain_openai==0.3.30

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# Import core libraries
import os                                                                       # Interact with the operating system (e.g., set environment variables)
import json                                                                     # Read/write JSON data
import requests  # type: ignore                                                 # Make HTTP requests (e.g., API calls); ignore type checker

# Import libraries for working with PDFs and OpenAI
from langchain_community.document_loaders import PyMuPDFLoader                            # Load and extract text from PDF files
# from langchain_community.document_loaders import PyPDFLoader                    # Load and extract text from PDF files
from openai import OpenAI                                                       # Access OpenAI's models and services

# Import libraries for processing dataframes and text
import tiktoken                                                                 # Tokenizer used for counting and splitting text for models
import pandas as pd                                                             # Load, manipulate, and analyze tabular data

# Import LangChain components for data loading, chunking, embedding, and vector DBs
from langchain.text_splitter import RecursiveCharacterTextSplitter              # Break text into overlapping chunks for processing
from langchain.embeddings.openai import OpenAIEmbeddings                        # Create vector embeddings using OpenAI's models  # type: ignore
from langchain.vectorstores import Chroma                                       # Store and search vector embeddings using Chroma DB  # type: ignore

from datasets import Dataset                                                    # Used to structure the input (questions, answers, contexts etc.) in tabular format
from langchain_openai import ChatOpenAI                                         # This is needed since LLM is used in metric computation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Question Answering using LLM

### OpenAI API Calling



> **Note:** <br> Make sure to create a `config.json` file in your project directory containing your OpenAI credentials in the following format:
<br><br>```{"API_KEY": "your_openai_api_key_here","OPENAI_API_BASE": "your_api_base"}```<br><br>
Replace the placeholder with your actual API key. This file allows your script to securely load API configuration details without hardcoding them directly into the code. </br>


In [ ]:
# Load the JSON file and extract values
file_name = "config.json"                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    OPENAI_API_KEY = config.get("OPENAI_API_KEY")                                             # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")                             # Extract the OpenAI base URL from the config

# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY                                          # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE                                 # Set API base URL as environment variable

# Initialize OpenAI client
client = OpenAI()                                                               # Create an instance of the OpenAI client

### Response

In [ ]:
# Define a function to get a response
def response(user_prompt, max_tokens=500, temperature=0.2, top_p=0.95):   # Complete the code to set default paramenters
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4o-mini",  # Complete the code by specifying the model to be used.
        messages=[
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content                                # Return the text content from the model's reply

### Question 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
question_1 = "What is the protocol for managing sepsis in a critical care unit?"
base_prompt_response_1=response(question_1)
base_prompt_response_1

"Managing sepsis in a critical care unit involves a systematic approach that focuses on early recognition, timely intervention, and ongoing monitoring. The following is a general protocol based on current guidelines, such as those from the Surviving Sepsis Campaign:\n\n### 1. **Early Recognition**\n   - **Identify Symptoms**: Look for signs of infection (fever, chills, tachycardia, tachypnea) and organ dysfunction (altered mental status, hypotension, oliguria).\n   - **Use Screening Tools**: Utilize tools like the qSOFA (quick Sequential Organ Failure Assessment) or SIRS (Systemic Inflammatory Response Syndrome) criteria to identify patients at risk.\n\n### 2. **Initial Assessment**\n   - **Obtain History and Physical Exam**: Assess for potential sources of infection and evaluate the patient's clinical status.\n   - **Laboratory Tests**: Obtain blood cultures before starting antibiotics, along with other relevant tests (CBC, lactate, renal function, liver function tests).\n\n### 3. **F

### Question 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
question_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it" #Complete the code to define the question #2
base_prompt_response_2=response(question_2) #Complete the code to pass the user input
base_prompt_response_2

'Common symptoms of appendicitis include:\n\n1. **Abdominal Pain**: Typically starts around the navel and then moves to the lower right abdomen.\n2. **Nausea and Vomiting**: Often follows the onset of abdominal pain.\n3. **Loss of Appetite**: A decrease in the desire to eat.\n4. **Fever**: Mild fever may develop as the body responds to inflammation.\n5. **Constipation or Diarrhea**: Changes in bowel habits can occur.\n6. **Abdominal Swelling**: In some cases, the abdomen may become swollen or tender.\n\nAppendicitis is not typically treated with medication alone. The standard treatment is surgical removal of the appendix, a procedure known as an **appendectomy**. This can be performed using two main techniques:\n\n1. **Open Appendectomy**: A larger incision is made in the lower right abdomen to remove the appendix.\n2. **Laparoscopic Appendectomy**: This is a minimally invasive procedure where several small incisions are made, and the appendix is removed using a camera and specialized 

### Question 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
question_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it" #Complete the code to define the question #3
base_prompt_response_3=response(question_3) #Complete the code to pass the user input
base_prompt_response_3

'Sudden patchy hair loss, often referred to as alopecia areata, can manifest as localized bald spots on the scalp or other areas of the body. Here are some effective treatments and solutions, as well as potential causes of this condition:\n\n### Possible Causes:\n1. **Autoimmune Disorders**: Alopecia areata is believed to be an autoimmune condition where the immune system mistakenly attacks hair follicles.\n2. **Genetic Factors**: A family history of alopecia or other autoimmune diseases may increase the risk.\n3. **Stress**: Physical or emotional stress can trigger hair loss in some individuals.\n4. **Hormonal Changes**: Hormonal fluctuations, such as those occurring during pregnancy or menopause, can contribute to hair loss.\n5. **Nutritional Deficiencies**: Lack of essential nutrients, such as iron, zinc, or vitamin D, can affect hair health.\n6. **Infections**: Fungal infections like tinea capitis can cause patchy hair loss.\n7. **Other Medical Conditions**: Conditions like thyroid

### Question 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
question_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function" #Complete the code to define the question #4
base_prompt_response_4=response(question_4) #Complete the code to pass the user input
base_prompt_response_4

'The treatment for a person who has sustained a physical injury to brain tissue, such as a traumatic brain injury (TBI), can vary widely depending on the severity of the injury, the specific areas of the brain affected, and the resulting impairments. Here are some common approaches to treatment:\n\n### Immediate Medical Treatment\n1. **Emergency Care**: Initial treatment may involve stabilizing the patient, ensuring adequate oxygen supply, and managing intracranial pressure.\n2. **Surgery**: In some cases, surgical intervention may be necessary to remove blood clots, repair skull fractures, or relieve pressure on the brain.\n\n### Rehabilitation\n1. **Physical Therapy**: Helps improve motor skills, balance, and coordination.\n2. **Occupational Therapy**: Focuses on helping the individual regain the ability to perform daily activities and improve fine motor skills.\n3. **Speech and Language Therapy**: Assists with communication difficulties and swallowing issues.\n4. **Neuropsychologica

### Storing the generated outputs from the base prompt


In [ ]:
# Create the DataFrame
result_df = pd.DataFrame({
    "questions": [question_1, question_2, question_3, question_4],
    "base_prompt_responses": [base_prompt_response_1, base_prompt_response_2, base_prompt_response_3, base_prompt_response_4]})

# Display the DataFrame
result_df.head()

,questions,base_prompt_responses
0,What is the protocol for managing sepsis in a ...,Managing sepsis in a critical care unit involv...
1,"What are the common symptoms for appendicitis,...",Common symptoms of appendicitis include:\n\n1....
2,What are the effective treatments or solutions...,"Sudden patchy hair loss, often referred to as ..."
3,What treatments are recommended for a person w...,The treatment for a person who has sustained a...


**Observations:**

The baseline LLM generated clinically coherent and well structured responses across all questions. The low temperature setting contributed to consistent and focused outputs. However, responses remained high level and generalized, without source attribution or explicit grounding in a medical reference. While no overt inaccuracies were observed, the absence of contextual grounding introduces potential hallucination risk and limits verifiability.

## Question Answering using LLM with Prompt Engineering

In the next step, we will use prompt engineering to check the effect of a more detailed and well-engineered prompt on the output of the model.

In [ ]:
# Define the system prompt for the model (Prompt Engineering only, no RAG context)
system_prompt = """
You are an AI medical assistant designed to support healthcare professionals by answering clinical questions with accurate, structured, and safety minded responses.

User input will include a clinical question. Your task is to provide a best effort answer using general medical knowledge while clearly stating uncertainty when appropriate.

When crafting your response
- Prioritize clinical accuracy and patient safety over creativity
- Use clear, direct medical language suitable for a clinician audience
- Use a structured format (bullet points or short sections) when helpful
- If the question asks for a protocol or management approach, present stepwise actions (assessment, key tests, initial management, escalation)
- If the question asks for symptoms, causes, or treatments, separate them into distinct sections
- Do not invent precise medication dosing, thresholds, or timelines. Only include these if you are confident they are standard and non controversial. Otherwise, say that specifics depend on local guidelines or specialist input
- If you are uncertain or the question requires a specific guideline citation, respond with: "Sorry, this is out of my knowledge base."

Safety and scope rules
- Do not provide diagnosis for a specific real patient
- Do not replace professional medical judgement
- If the condition discussed may be life threatening, advise urgent medical evaluation rather than giving detailed emergency instructions
- When discussing management, distinguish clearly between standard first-line approaches and alternative options

Please adhere to the following response guidelines
- Keep answers concise but clinically useful
- Avoid speculation and avoid unnecessary detail
- Clearly label any assumptions
- Focus only on what is directly relevant to the question
- If the question cannot be answered safely without a source, respond with: "Sorry, this is out of my knowledge base."

Here is an example of how to structure your response:

Answer:
[Direct answer in structured sections]

Clinical notes:
[Key considerations, red flags, or when to escalate]

Limitations:
[1 to 2 lines on what may vary by guideline or requires confirmation]

"""


### Defining the function to Generate a Response From the LLM

In [ ]:
# Define a function to get a response from the OpenAI chat model
def response(system_prompt, user_prompt, max_tokens=500, temperature=0.2, top_p=0.95):  # Complete the code to set default paramenters
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4o-mini",                                                        # Complete the code by specifying the model to be used.
        messages=[
            {"role": "system", "content": system_prompt},                       # System prompt sets the assistant's behavior
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output (0 = deterministic)
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content                                # Return the text content from the model's reply

### Question 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
response_with_prompt_eng_1=response(system_prompt,question_1)
response_with_prompt_eng_1

'Answer:\n**Management Protocol for Sepsis in a Critical Care Unit:**\n\n1. **Initial Assessment:**\n   - Evaluate vital signs (heart rate, blood pressure, respiratory rate, temperature).\n   - Assess for signs of organ dysfunction (e.g., altered mental status, oliguria, elevated lactate).\n\n2. **Key Tests:**\n   - Obtain blood cultures before starting antibiotics.\n   - Complete blood count, basic metabolic panel, liver function tests, coagulation profile.\n   - Lactate level measurement.\n   - Consider imaging studies to identify the source of infection.\n\n3. **Initial Management:**\n   - **Fluid Resuscitation:** Administer intravenous fluids (crystalloids) to restore hemodynamic stability.\n   - **Antibiotic Therapy:** Initiate broad-spectrum intravenous antibiotics within 1 hour of sepsis recognition.\n   - **Vasopressors:** If hypotension persists despite adequate fluid resuscitation, initiate vasopressors (e.g., norepinephrine) to maintain mean arterial pressure (MAP) ≥ 65 mmHg

### Question 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
response_with_prompt_eng_2=response(system_prompt,question_2) #Complete the code to pass the user prompt and system prompt
response_with_prompt_eng_2

'Answer:\n\n**Common Symptoms of Appendicitis:**\n- Abdominal pain (typically starting near the belly button and then shifting to the lower right abdomen)\n- Nausea and vomiting\n- Loss of appetite\n- Fever\n- Constipation or diarrhea\n- Abdominal swelling\n\n**Medical Management:**\n- Appendicitis is primarily a surgical condition. While antibiotics may be used in some cases (e.g., uncomplicated acute appendicitis), they are not a definitive cure. Surgical intervention is generally required.\n\n**Surgical Procedure:**\n- The standard surgical procedure for appendicitis is an appendectomy, which can be performed as:\n  - **Open Appendectomy:** A larger incision in the abdomen.\n  - **Laparoscopic Appendectomy:** A minimally invasive technique using small incisions and a camera.\n\nClinical notes:\n- Early diagnosis and treatment are crucial to prevent complications such as perforation or abscess formation.\n- Patients presenting with classic symptoms should be evaluated urgently.\n\nLi

### Question 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
response_with_prompt_eng_3=response(system_prompt,question_3) #Complete the code to pass the user prompt and system prompt
response_with_prompt_eng_3

'Answer:\n\n### Possible Causes:\n- **Alopecia Areata**: An autoimmune condition leading to sudden hair loss in patches.\n- **Stress**: Physical or emotional stress can trigger hair loss.\n- **Infections**: Fungal infections like tinea capitis can cause localized hair loss.\n- **Nutritional Deficiencies**: Deficiencies in iron, zinc, or biotin may contribute.\n- **Thyroid Disorders**: Hypothyroidism or hyperthyroidism can lead to hair loss.\n- **Traction Alopecia**: Hair styles that pull on the hair can cause localized loss.\n\n### Effective Treatments:\n- **Topical Corticosteroids**: First-line treatment for alopecia areata; reduces inflammation.\n- **Minoxidil**: May promote hair regrowth; often used in conjunction with other treatments.\n- **Intralesional Corticosteroids**: Injections directly into the bald spots can be effective.\n- **Immunotherapy**: For extensive alopecia areata, treatments like diphencyprone (DPCP) can be used.\n- **Antifungal Treatment**: If a fungal infection 

### Question 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
response_with_prompt_eng_4=response(system_prompt,question_4) #Complete the code to pass the user prompt and system prompt
response_with_prompt_eng_4

"Answer:\n**Treatment Approaches for Brain Injury:**\n\n1. **Initial Assessment and Stabilization:**\n   - Conduct a thorough neurological examination.\n   - Monitor vital signs and neurological status.\n   - Obtain imaging studies (CT or MRI) to assess the extent of injury.\n\n2. **Acute Management:**\n   - Ensure airway, breathing, and circulation (ABCs) are stable.\n   - Administer oxygen if hypoxic.\n   - Manage intracranial pressure (ICP) if elevated; consider mannitol or hypertonic saline as per local protocols.\n\n3. **Symptomatic Treatment:**\n   - Pain management with analgesics.\n   - Anticonvulsants if seizures occur.\n   - Consider medications for specific symptoms (e.g., antidepressants for mood changes).\n\n4. **Rehabilitation:**\n   - Initiate physical, occupational, and speech therapy as soon as feasible.\n   - Tailor rehabilitation goals based on the individual's impairments and functional needs.\n\n5. **Long-term Management:**\n   - Regular follow-up with a neurologis

### Storing the generated outputs from the structured prompts

In [ ]:
# Add the results to a new column in the DataFrame
result_df['responses_with_prompt_eng'] = [response_with_prompt_eng_1, response_with_prompt_eng_2, response_with_prompt_eng_3, response_with_prompt_eng_4]

# Display the DataFrame
result_df.head()

,questions,base_prompt_responses,responses_with_prompt_eng
0,What is the protocol for managing sepsis in a ...,Managing sepsis in a critical care unit involv...,Answer:\n**Management Protocol for Sepsis in a...
1,"What are the common symptoms for appendicitis,...",Common symptoms of appendicitis include:\n\n1....,Answer:\n\n**Common Symptoms of Appendicitis:*...
2,What are the effective treatments or solutions...,"Sudden patchy hair loss, often referred to as ...",Answer:\n\n### Possible Causes:\n- **Alopecia ...
3,What treatments are recommended for a person w...,The treatment for a person who has sustained a...,Answer:\n**Treatment Approaches for Brain Inju...


**Observations**:

Compared to the baseline LLM, the prompt engineered LLM produced more structured, clinically sequenced, and safety-oriented responses. Explicit formatting instructions improved clarity, stepwise protocol flow, and management hierarchy.

After further tightening the prompt by adding “Focus only on what is directly relevant to the question” and “Distinguish clearly between standard first-line management and alternative options,” the model demonstrated improved contextual discipline, reduced overgeneralization, and clearer prioritization of first-line versus alternative approaches.

However, despite these structural and organizational improvements, responses remained generalized and were not grounded in a specific medical reference. The outputs lacked citation support and protocol-level specificity, limiting verifiability and measurable context precision.

This demonstrates that while prompt engineering enhances structure and safety framing, it cannot ensure evidence anchoring or guideline alignment in the absence of retrieval. These limitations justify the integration of retrieval-augmented generation to improve groundedness, traceability, and contextual reliability.

## Data Preparation for RAG

### Loading the Data

In [ ]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
manual_pdf_path = "/content/drive/MyDrive/Colab Notebooks/medical_diagnosis_manual.pdf" #Complete the code to define the file name

In [ ]:
pdf_loader = PyMuPDFLoader(manual_pdf_path)

In [ ]:
manual = pdf_loader.load()

### Data Overview

#### Checking the first 5 pages

In [ ]:
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(manual[i].page_content,end="\n")

Page Number : 1
renee.manzari@gmail.com
LMQV6N40CT
This file is meant for personal use by renee.manzari@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 2
renee.manzari@gmail.com
LMQV6N40CT
This file is meant for personal use by renee.manzari@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .......................................................................................................................

### Data Chunking

#### Chunk the PDF into Manageable Text Sections Using a Token-Based Splitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512, #Complete the code to define the chunk size
    chunk_overlap=64 #Complete the code to define the chunk overlap
)

#### Split the Loaded PDF into Chunks for Further Processing

In [ ]:
document_chunks = pdf_loader.load_and_split(text_splitter)

#### Check the Number of Chunks Created

In [ ]:
len(document_chunks)

8781

### Generate Vector Embeddings for Text Chunks Using OpenAI

In [ ]:
# Initialize the OpenAI Embeddings model with API credentials
embedding_model = OpenAIEmbeddings(
    openai_api_key=OPENAI_API_KEY,                                                     # Your OpenAI API key for authentication
    openai_api_base=OPENAI_API_BASE # The OpenAI API base URL endpoint
)

# Generate embeddings (vector representations) for the first two document chunks
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)      # Embedding for chunk 0
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)      # Embedding for chunk 1

# Check and print the dimension (length) of the embedding vector
print("Dimension of the embedding vector ", len(embedding_1))                   # Typically 1536 or 2048 depending on model

/tmp/ipython-input-317147261.py:2: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embedding_model = OpenAIEmbeddings(


Dimension of the embedding vector  1536


In [ ]:
# Verify if both embeddings have the same dimension (should be True)
len(embedding_1) == len(embedding_2)

# Return/display the two embedding vectors for further inspection or use
embedding_1, embedding_2

([-0.015376257367345636,
  0.002842827576053951,
  0.010522398569091734,
  -0.025960271309009617,
  -0.015170875586072646,
  0.010056866717804145,
  -0.02330399766350897,
  -0.020880491340636034,
  -0.04069299779118156,
  -0.015143490976040536,
  0.03948808985093051,
  0.009878868615240656,
  0.01708777314260983,
  0.013082826079479998,
  -0.0006448135492386715,
  0.0014094332539685802,
  0.011829996934317976,
  -0.017690227112735355,
  0.017950377182749954,
  0.0005780644354003519,
  0.006003997052779428,
  0.006514028895369194,
  -0.0181420675903295,
  0.013247131877027434,
  -0.011686229594294622,
  -0.027589633254177483,
  0.025946579003993563,
  -0.026069807886492834,
  0.018388525355328044,
  -0.031026357109721923,
  0.009851484936531155,
  0.005665117160245125,
  -0.013308746318277071,
  -0.016896082735030283,
  -0.014472577343479957,
  0.019949427638060856,
  0.027658094779257757,
  -0.0011030718873554495,
  0.02502920574378922,
  -0.012391373989395337,
  0.023933836243666604,


### Vector Database

#### Setup Vector Database Directory

In [ ]:
out_dir = 'manual_diagnosis_manual.pdf'    # complete the code to define the name of the vector database

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [ ]:
# Building the vector store and saving it to disk for future use
# Initialize an empty Chroma vectorstore
vectorstore = Chroma(persist_directory=out_dir, embedding_function=embedding_model)

# Add documents in batches to avoid exceeding token limits
batch = 200  # number of chunks to add at a time
for i in range(0, len(document_chunks), batch):
    vectorstore.add_documents(document_chunks[i:i+batch])

vectorstore.persist()


/tmp/ipython-input-3379083442.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=out_dir, embedding_function=embedding_model)
/tmp/ipython-input-3379083442.py:10: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


#### Load Vector Database

In [ ]:
vectorstore = Chroma(
    persist_directory=out_dir,
    embedding_function=embedding_model
)

#### Explore Vector Database and Perform Searches

In [ ]:
vectorstore.embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x79dd098d8a40>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x79dd09660c20>, model='text-embedding-ada-002', deployment='text-embedding-ada-002', openai_api_version='', openai_api_base='https://aibe.mygreatlearning.com/openai/v1', openai_api_type='', openai_proxy='', embedding_ctx_length=8191, openai_api_key='gl-U2FsdGVkX1+TW2TWRbbxdoVLctC8YO+WJPFK9fcMf+kzWQqfYa6u270OwC5lymBi', openai_organization=None, allowed_special=set(), disallowed_special='all', chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None)

write an instruction on what to do in next cell

In [ ]:
vectorstore.similarity_search("What is the protocol for managing sepsis in a critical care unit?",k=4) #Complete the code to pass a query and an appropriate k value

[Document(metadata={'moddate': '2026-02-13T02:21:09+00:00', 'modDate': 'D:20260213022109Z', 'creationDate': 'D:20120615054440Z', 'creator': 'Atop CHM to PDF Converter', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'source': '/content/drive/MyDrive/Colab Notebooks/medical_diagnosis_manual.pdf', 'keywords': '', 'total_pages': 4114, 'subject': '', 'format': 'PDF 1.7', 'author': '', 'trapped': '', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'creationdate': '2012-06-15T05:44:40+00:00', 'file_path': '/content/drive/MyDrive/Colab Notebooks/medical_diagnosis_manual.pdf', 'page': 2456}, page_content="Parenteral antibiotics should be given after specimens of blood, body fluids, and wound sites have been\ntaken for Gram stain and culture. Very prompt empiric therapy, started immediately after suspecting\nsepsis, is essential and may be lifesaving. Antibiotic selection requires an educated guess based on the\nsuspected source, clinical setting, knowledge or sus

### Retrieval and Response Generation using Vector Search

#### Convert Vector Database into a Retriever and Retrieve Relevant Documents

In [ ]:

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,              # Number of final documents returned to the LLM
        "fetch_k": 20,       # Number of documents initially fetched before diversification
        "lambda_mult": 0.5   # Balance between relevance and diversity
    }
)

### System and User Prompt Template

Prompts guide the model to generate accurate responses. Here, we define two parts:

    1. The system message describing the assistant's role.
    2. A user message template including context and the question.

In [ ]:
# Complete the code to define the RAG System Prompt
qna_system_message = """
You are an AI medical assistant designed to support healthcare professionals.
You must answer clinical questions using ONLY the provided excerpts from
The Merck Manual of Diagnosis and Therapy.

User input will include context that begins with:

###Context

The context contains retrieved excerpts from The Merck Manual, often including section headings and page numbers.

Core rules (mandatory):
- Use ONLY facts explicitly stated in the provided context.
- Do NOT use prior knowledge, medical training data, assumptions, or outside information.
- Do NOT guess, infer, extrapolate, or add details not explicitly supported by the context.
- If the context is empty, unrelated, or does not contain enough information to answer safely, respond exactly with:
  "Sorry, this is out of my knowledge base."
- Focus only on what is directly relevant to the question.
- When discussing management, distinguish between first-line and alternative options ONLY if the context explicitly describes them.
- If the context contains multiple excerpts, synthesize them into one answer WITHOUT adding any new information.

Evidence and citation behavior:
- Every key clinical statement must be supported by the provided context.
- Cite sources using Merck Manual page number(s) and section title(s) when available.
- If page numbers are not available, cite any section heading or identifier provided in the context.

Response guidelines:
- Provide clear, concise, clinically structured answers.
- Use bullet points or numbered steps for protocols and management.
- Do not introduce new differentials, tests, treatments, dosing details, thresholds, timelines, or clinical guidance unless they are explicitly stated in the context.

Output format (follow exactly):

Answer:
[Clinically structured answer based strictly on the context.]

Source:
[Merck Manual page number(s) and section title(s) used.]
"""


In [ ]:
qna_user_message_template = """
###Context
The following excerpts are retrieved from The Merck Manual of Diagnosis and Therapy and are relevant to the clinical question below:
{context}

###Question
{question}

Please answer using ONLY the information provided in the context above.
"""
#Complete the code to define the user message

### Response Function

In [ ]:
def generate_rag_response(user_input,k=4,max_tokens=500,temperature=0,top_p=0.95):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    # Generate the response
    try:
        response = client.chat.completions.create(
        model="gpt-4o-mini",   # Complete the code by specifying the model to be used.
        messages=[
            {"role": "system", "content": qna_system_message},
            {"role": "user", "content": user_message}
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p
        )
        # Extract and print the generated text from the response
        response = response.choices[0].message.content.strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Question 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
response_with_rag_1 = generate_rag_response(question_1)
response_with_rag_1

/tmp/ipython-input-403186717.py:4: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)


'Answer:\n- **Initial Management:**\n  - Obtain specimens of blood, body fluids, and wound sites for Gram stain and culture before starting antibiotics.\n  - Initiate very prompt empiric therapy immediately after suspecting sepsis, as it may be lifesaving.\n\n- **Antibiotic Selection:**\n  - Choose antibiotics based on:\n    - Suspected source of infection\n    - Clinical setting\n    - Knowledge or suspicion of causative organisms and sensitivity patterns\n    - Previous culture results\n  - One regimen for septic shock of unknown cause includes:\n    - Gentamicin or tobramycin 5.1 mg/kg IV once/day plus a 3rd-generation cephalosporin (options include cefotaxime 2 g q 6 to 8 h, ceftriaxone 2 g once/day, or ceftazidime 2 g IV q 8 h if Pseudomonas is suspected).\n    - Alternatively, use ceftazidime plus a fluoroquinolone (e.g., ciprofloxacin).\n    - Vancomycin must be added if resistant staphylococci or enterococci are suspected.\n    - Include a drug effective against anaerobes (e.g.

### Question 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
response_with_rag_2 = generate_rag_response(question_2) #Complete the code to pass the user input
response_with_rag_2

"Answer:\nCommon symptoms of appendicitis include:\n- Epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia.\n- Pain that shifts to the right lower quadrant after a few hours.\n- Pain increases with cough and motion.\n- Right lower quadrant direct and rebound tenderness at McBurney's point.\n- Additional signs may include Rovsing sign, psoas sign, and obturator sign.\n- Low-grade fever (rectal temperature 37.7 to 38.3° C [100 to 101° F]) is common.\n\nAppendicitis cannot be cured via medicine alone; it requires surgical treatment. The recommended surgical procedure is either an open or laparoscopic appendectomy.\n\nSource:\nThe Merck Manual of Diagnosis & Therapy, 19th Edition, Chapter 11. Acute Abdomen & Surgical Gastroenterology."

### Question 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
response_with_rag_3 = generate_rag_response(question_3) #Complete the code to pass the user input
response_with_rag_3

'Answer:\n- **Effective Treatments for Sudden Patchy Hair Loss (Alopecia Areata)**:\n  - Treatment options are not explicitly detailed in the provided context for alopecia areata. However, it is noted that microscopic hair examination or scalp biopsy may be indicated when the diagnosis is in doubt.\n\n- **Possible Causes of Sudden Patchy Hair Loss**:\n  - Alopecia areata is characterized by sudden patchy hair loss in individuals without obvious skin or systemic disorders.\n  - It is thought to be an autoimmune disorder affecting genetically susceptible individuals.\n  - Environmental triggers may also play a role in the onset of alopecia areata.\n\nSource:\nMerck Manual, Chapter 86. Hair Disorders.'

### Question 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
response_with_rag_4 = generate_rag_response(question_4) #Complete the code to pass the user input
response_with_rag_4

'Answer:\n- Initial treatment for traumatic brain injury (TBI) includes:\n  - Ensuring a reliable airway.\n  - Maintaining adequate ventilation, oxygenation, and blood pressure.\n  - Surgical intervention may be necessary for severe injuries to monitor and treat intracranial pressure, decompress the brain, or remove hematomas.\n- Rehabilitation is needed when neurologic deficits persist, ideally through a team approach that includes:\n  - Physical therapy.\n  - Occupational therapy.\n  - Speech therapy.\n  - Skill-building activities.\n  - Counseling to address social and emotional needs.\n- For patients in a coma exceeding 24 hours, a prolonged period of rehabilitation focusing on cognitive and emotional areas is often required.\n- Early intervention by rehabilitation specialists is crucial for maximal functional recovery, including:\n  - Prevention of secondary disabilities (e.g., pressure ulcers, joint contractures).\n  - Prevention of pneumonia.\n  - Family education.\n- Extensive 

### Storing the RAG system outputs


In [ ]:
# Add the results to a new column in the DataFrame
result_df['responses_with_RAG'] = [response_with_rag_1, response_with_rag_2, response_with_rag_3, response_with_rag_4]

# Display the DataFrame
result_df.head()

,questions,base_prompt_responses,responses_with_prompt_eng,responses_with_RAG
0,What is the protocol for managing sepsis in a ...,Managing sepsis in a critical care unit involv...,Answer:\n**Management Protocol for Sepsis in a...,Answer:\n- **Initial Management:**\n - Obtain...
1,"What are the common symptoms for appendicitis,...",Common symptoms of appendicitis include:\n\n1....,Answer:\n\n**Common Symptoms of Appendicitis:*...,Answer:\nCommon symptoms of appendicitis inclu...
2,What are the effective treatments or solutions...,"Sudden patchy hair loss, often referred to as ...",Answer:\n\n### Possible Causes:\n- **Alopecia ...,Answer:\n- **Effective Treatments for Sudden P...
3,What treatments are recommended for a person w...,The treatment for a person who has sustained a...,Answer:\n**Treatment Approaches for Brain Inju...,Answer:\n- Initial treatment for traumatic bra...


Observation Experiment 1

Using the base RAG configuration (chunk size 512, overlap 64, k=3, similarity retriever, temperature=0), the model produced concise, clinically structured, and clearly grounded responses directly supported by excerpts from The Merck Manual. The retrieved chunks were highly relevant, and the answers reflected extractive synthesis rather than generative expansion. This setup minimized hallucination, reduced overgeneralization, and improved traceability through source citation. The chosen chunk size preserved coherent clinical sections, overlap prevented fragmentation, and k=3 balanced precision with sufficient coverage while avoiding off-topic retrieval. Temperature set to 0 further strengthened determinism and faithfulness, resulting in responses that were accurate but less expansive.

Compared to the baseline and prompt-engineered LLM outputs, the RAG responses were shorter, more conservative, and strictly evidence-bound. While the baseline and prompt-engineered models provided broader explanations and smoother narrative flow, they relied on internal model knowledge and were not verifiable against a specific source. In contrast, the RAG setup significantly improved groundedness, context precision, and faithfulness at the expense of some explanatory depth. This demonstrates the tradeoff between generative richness and evidence-based reliability, highlighting the value of retrieval-augmented generation for safety-critical clinical applications.


Observation 2

In Experiment 2, increasing k from 3 to 4 while maintaining chunk size 512, overlap 64, similarity retrieval, and temperature 0 resulted in slightly broader contextual coverage without sacrificing groundedness. Responses remained strictly extractive and verifiable against the Merck Manual, but showed modest improvements in completeness, particularly for multi-step management questions such as sepsis and traumatic brain injury. The chunking configuration continued to preserve coherent clinical sections, and no fragmentation issues were observed. The similarity retriever with k=4 expanded supporting details while maintaining focus, though it introduced a slightly higher theoretical risk of adjacent-section retrieval compared to k=3.

Compared to the base LLM and prompt-engineered LLM, the fine-tuned RAG setup remained more conservative, structured, and evidence-bound. While the non-RAG models provided broader clinical reasoning and more expansive explanations, they lacked source traceability and verifiability. The RAG configuration in Experiment 2 demonstrates a stronger balance between contextual completeness and faithfulness, improving protocol coverage while maintaining deterministic, grounded responses suitable for safety-critical medical support.


Observation 3

In Experiment 3, increasing k from 4 to 5 while keeping chunk size at 512 and overlap at 64 led to broader contextual retrieval and slightly more comprehensive answers. Responses incorporated more surrounding material from the Merck Manual, which improved coverage of clinical protocols and treatment details, particularly for sepsis and alopecia. However, this increase in recall came with a mild reduction in precision. Answers became slightly longer and more synthesized rather than tightly extractive, reflecting greater contextual blending across retrieved chunks. The chunking strategy remained stable and coherent, but the higher k value increased the amount of material the model had to reconcile.

Compared to the base LLM and prompt-engineered LLM, the RAG system with k=5 remained clearly more grounded, verifiable, and faithful to source material, with no hallucinated expansions. However, unlike the prompt-engineered LLM, it did not provide broader clinical reasoning beyond the retrieved text. Relative to earlier RAG experiments with lower k values, this configuration demonstrates the tradeoff between recall and contextual discipline: k=5 improves completeness for complex questions but slightly reduces answer sharpness and conciseness.


Observation 4

In Experiment 4, reducing the chunk overlap from 64 to 20 while keeping chunk size at 512 and k=4 maintained strong grounding and faithfulness but slightly reduced contextual continuity. The responses remained accurate, structured, and strictly based on the Merck Manual, with no hallucinations due to temperature being set to 0. However, the lower overlap weakened the contextual buffering between adjacent sections, leading to minor fragmentation in multi-step protocols such as sepsis and TBI management. Completeness was largely preserved because k=4 retrieved sufficient supporting chunks, but transitions between related clinical steps were slightly less fluid compared to overlap 64.

Compared to the base LLM and prompt-engineered LLM, this RAG configuration continued to produce more verifiable, extractive, and source-grounded responses, avoiding speculative reasoning or generalization. While it remained less expansive and interpretive than the prompt-engineered LLM, it provided higher traceability and clinical precision. Relative to overlap 64, overlap 20 reduced redundancy but offered no meaningful gain in accuracy, suggesting that overlap 64 may provide better contextual stability for dense medical manuals without increasing hallucination risk.


Observation 5

In Experiment 5, reducing the chunk size from 512 to 400 while keeping overlap at 64 and k=4 increased text granularity but slightly reduced protocol-level cohesion. Because smaller chunks capture narrower context windows, multi-step clinical workflows such as sepsis management and TBI treatment became more segmented, requiring the retriever to assemble more distributed context. While overlap 64 helped maintain continuity, responses appeared marginally more extractive and less holistically structured compared to chunk size 512. Importantly, faithfulness and grounding remained strong due to similarity retrieval and temperature set to 0.

Compared to the base LLM and prompt-engineered LLM, this RAG configuration continued to provide more verifiable and source-grounded responses, eliminating speculative expansion. However, relative to the earlier 512-chunk experiments, chunk size 400 did not improve precision or completeness and slightly increased fragmentation risk. This suggests that for dense, structured medical manuals like Merck, chunk size 512 offers better contextual stability without sacrificing retrieval accuracy.


Observation 6

In Experiment 6, switching the retriever from similarity to MMR while keeping chunk size at 512, overlap at 64, and k=4 improved contextual diversity without sacrificing grounding or precision. By using fetch_k=20 and lambda=0.5, MMR selected chunks that were both relevant and non-redundant, resulting in more balanced coverage of multi-step clinical protocols such as sepsis and TBI management. Compared to similarity retrieval, responses demonstrated better integration of distinct management phases and reduced repetition of adjacent content, while still remaining fully extractive and strictly grounded in the Merck Manual due to temperature being set to 0.

Relative to the base LLM and prompt-engineered LLM, the MMR-based RAG setup remained more verifiable, source-grounded, and resistant to hallucination. While it was less interpretive and expansive than the prompt-engineered LLM, it provided stronger traceability and contextual discipline. Among all tested configurations, this setup achieved the best balance between precision, contextual completeness, and retrieval diversity, making it the most robust configuration for safety-critical medical question answering using a dense reference corpus like the Merck Manual.


Conclusion

Across six controlled experiments, the optimal RAG configuration for a dense, protocol-heavy medical corpus like the Merck Manual is:

512 chunk size, 64 overlap, k=4, MMR retrieval, temperature=0.


This setup delivers the strongest balance between:

Precision (minimal irrelevant retrieval)

Contextual completeness (full protocol representation)

Retrieval discipline (reduced redundancy via MMR)

Faithfulness (strict grounding in source text)

Structural coherence (preserved multi-step clinical workflows)


Compared to baseline and prompt-engineered LLMs, the optimized RAG system:

Eliminates hallucinated generalizations

Provides verifiable, source-grounded clinical guidance

Reduces speculative expansion

Maintains safety-aligned deterministic behavior

Among all experiments, the MMR configuration demonstrated the most robust balance between recall and precision, making it the most appropriate design for safety-critical medical decision support systems using authoritative reference manuals.

This final configuration prioritizes reliability and traceability over expansiveness — a necessary tradeoff in clinical AI systems

## Output Evaluation using LLM as a Judge

---



#### **Defining required System Prompts**

In [ ]:
groundedness_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer, and an AI generated answer.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
1 - Not followed at all
2 - Followed to a limited extent
3 - Followed to a good extent
4 - Followed mostly
5 - Followed completely

Metric:
Groundedness measures whether the answer is derived only from the information presented in the context.
Consider whether all factual claims in the answer are directly supported by the context.
The answer must not introduce information, reasoning, or conclusions that are not explicitly stated or clearly supported in the context.

Instructions (follow this exact format):
1. Steps to evaluate the answer as per the metric:
- Identify key factual claims made in the answer.
- Compare each claim against the provided context.
- Determine whether any part of the answer includes unsupported or external information.
- Verify that no inference beyond the context is introduced.

2. Step-by-step explanation:
- Explain whether each major claim in the answer is supported by the context.
- Identify any unsupported or hallucinated elements, if present.

3. Evaluation of the extent to which the metric is followed:
- Provide a brief justification of how well the answer adheres strictly to the context.

4. Score assignment:
- Assign a score based on how fully the answer is grounded in the context.

Return the score on the LAST line only in a dictionary format (not JSON), like:
{groundedness_score:4}
"""

#Complete the code to define the prompt to evaluate groundedness

In [ ]:
relevance_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented with a question, context used by the AI system to generate the answer, and an AI generated answer.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
1 - Not followed at all
2 - Followed to a limited extent
3 - Followed to a good extent
4 - Followed mostly
5 - Followed completely

Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context.
Consider whether all and only the important aspects of the question are addressed in the answer.
The answer should stay focused on what was asked and avoid unnecessary or tangential information.

Instructions (follow this exact format):

1. Steps to evaluate the answer as per the metric:
- Identify the main components of the question.
- Determine which aspects of the question require a response.
- Compare those aspects with the content of the answer.
- Check whether any key parts of the question were omitted.
- Verify that the answer does not include irrelevant or unrelated information.

2. Step-by-step explanation:
- Explain whether the answer fully addresses the important aspects of the question.
- Identify any missing components.
- Identify any unnecessary or tangential information.

3. Evaluation of the extent to which the metric is followed:
- Provide a brief justification of how well the answer addresses the question.

4. Score assignment:
- Assign a score based on how completely and precisely the answer addresses the question.

Return the score on the LAST line only in a dictionary format (not JSON), like:
{relevance_score:4}
"""



In [ ]:
user_message_template = """
###Question
{question}

###Context (Retrieved from Merck Manual)
{context}

###Answer
{answer}
"""

Defining LLM as a Judge

In [ ]:
def generate_ground_relevance_response(user_input,response,  max_tokens=512, temperature=0, top_p=0.95):  # Complete the code to set default paramenters
    global qna_user_message_template

    context_for_query = [doc.page_content for doc in retriever.get_relevant_documents(user_input, k=4)]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=response)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=response)}
                [/INST]"""

    response_1 = client.chat.completions.create(
            model="gpt-4o-mini",   # Complete the code by specifying the model to be used.
            messages=[
                {"role": "user", "content": groundedness_prompt}
                ],
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p
            )

    response_2 = client.chat.completions.create(
            model="gpt-4o-mini",   # Complete the code by specifying the model to be used.
            messages=[
                {"role": "user", "content": relevance_prompt}
                ],
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p
            )

    return response_1.choices[0].message.content,response_2.choices[0].message.content

#### **Evaluation 1: Base Prompt Response Evaluation**

In [ ]:
# Question 1
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[0], response=result_df.base_prompt_responses[0], max_tokens=512)
llm_judge_base_ground_1 = int(ground.split('{groundedness_score:')[-1].replace('}', ''))
llm_judge_base_rel_1 = int(rel.split('{relevance_score:')[-1].replace('}', ''))
print(ground,end="\n\n")
print(rel)

1. Steps to evaluate the answer as per the metric:
- The answer outlines a protocol for managing sepsis in a critical care unit, including early recognition, initial assessment, fluid resuscitation, antibiotic therapy, vasopressor support, source control, and supportive care.
- Each claim made in the answer needs to be compared against the provided context to check for support or contradiction.

2. Step-by-step explanation:
- **Early Recognition**: The answer mentions identifying symptoms and using screening tools, which aligns with the general practices in critical care but is not explicitly stated in the context.
- **Initial Assessment**: The need for history, physical exam, and laboratory tests is a standard practice in critical care, but specific details about these practices are not provided in the context.
- **Fluid Resuscitation**: The answer suggests administering IV fluids and monitoring response, which is a common practice in sepsis management. However, the context does not p

In [ ]:
# Question 2
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[1], response=result_df.base_prompt_responses[1], max_tokens=512)  #Complete the code to calculate the groundedness and relevance score
llm_judge_base_ground_2 = int(ground.split('{groundedness_score:')[-1].replace('}', ''))
llm_judge_base_rel_2 = int(rel.split('{relevance_score:')[-1].replace('}', ''))
print(ground,end="\n\n")
print(rel)

1. Steps to evaluate the answer as per the metric:
- Identify key factual claims made in the answer.
- Compare each claim against the provided context.
- Determine whether any part of the answer includes unsupported or external information.
- Verify that no inference beyond the context is introduced.

2. Step-by-step explanation:
- The answer states that common symptoms of appendicitis include abdominal pain, nausea and vomiting, loss of appetite, fever, constipation or diarrhea, and abdominal swelling. The context supports the claims about abdominal pain (which starts around the navel and moves to the lower right abdomen), nausea and vomiting, loss of appetite, and mild fever. However, while changes in bowel habits (constipation or diarrhea) are mentioned in the context, abdominal swelling is not explicitly supported.
- The answer correctly states that appendicitis is not typically treated with medication alone and that the standard treatment is surgical removal of the appendix (appen

In [ ]:
# Question 3
ground, rel = generate_ground_relevance_response(
    user_input=result_df.questions[2],
    response=result_df.base_prompt_responses[2],
    max_tokens=512
)
llm_judge_base_ground_3 = int(ground.split('{groundedness_score:')[-1].replace('}', ''))
llm_judge_base_rel_3 = int(rel.split('{relevance_score:')[-1].replace('}', ''))
print(ground, end="\n\n")
print(rel)

1. Steps to evaluate the answer as per the metric:
- Identify key factual claims made in the answer.
- Compare each claim against the provided context.
- Determine whether any part of the answer includes unsupported or external information.
- Verify that no inference beyond the context is introduced.

2. Step-by-step explanation:
- The answer correctly identifies "alopecia areata" as a cause of sudden patchy hair loss, which is supported by the context.
- The potential causes listed (autoimmune disorders, genetic factors, stress, hormonal changes, nutritional deficiencies, infections, and other medical conditions) are all mentioned or implied in the context, particularly regarding autoimmune disorders and nutritional deficiencies.
- The treatments suggested (topical corticosteroids, minoxidil, intralesional corticosteroid injections, immunotherapy, oral medications, light therapy, nutritional support, stress management, and hairpieces or wigs) are mostly supported by the context. The c

In [ ]:
# Question 4
ground, rel = generate_ground_relevance_response(
    user_input=result_df.questions[3],
    response=result_df.base_prompt_responses[3],
    max_tokens=512
)
llm_judge_base_ground_4 = int(ground.split('{groundedness_score:')[-1].replace('}', ''))
llm_judge_base_rel_4 = int(rel.split('{relevance_score:')[-1].replace('}', ''))
print(ground, end="\n\n")
print(rel)

1. Steps to evaluate the answer as per the metric:
- The answer presents several key claims regarding the treatment of traumatic brain injury (TBI), including immediate medical treatment, rehabilitation therapies, medications, cognitive rehabilitation, lifestyle modifications, supportive care, and long-term management.
- Each claim should be compared against the provided context to determine if they are supported by the information in the context.
- The answer must not introduce any unsupported or external information.

2. Step-by-step explanation:
- **Immediate Medical Treatment**: The context mentions initial treatment involving stabilizing the patient and managing intracranial pressure, which supports the claims about emergency care and surgery.
- **Rehabilitation**: The context states that rehabilitation is needed when neurologic deficits persist and mentions physical, occupational, and speech therapy, which aligns with the answer's claims about rehabilitation therapies.
- **Medica

In [ ]:
# Create a DataFrame to store the base prompt evaluation results
base_prompt_evaluation_df = pd.DataFrame({
    "question": [question_1, question_2, question_3, question_4],
    "base_prompt_response": [result_df.base_prompt_responses[0], result_df.base_prompt_responses[1], result_df.base_prompt_responses[2], result_df.base_prompt_responses[3]],
    "groundedness_score": [llm_judge_base_ground_1, llm_judge_base_ground_2, llm_judge_base_ground_3, llm_judge_base_ground_4],
    "relevance_score": [llm_judge_base_rel_1, llm_judge_base_rel_2, llm_judge_base_rel_3, llm_judge_base_rel_4]
})

base_prompt_evaluation_df['groundedness_score'] = pd.to_numeric(base_prompt_evaluation_df['groundedness_score'], errors='coerce')
base_prompt_evaluation_df['relevance_score'] = pd.to_numeric(base_prompt_evaluation_df['relevance_score'], errors='coerce')

# Display the DataFrame
display(base_prompt_evaluation_df)

,question,base_prompt_response,groundedness_score,relevance_score
0,What is the protocol for managing sepsis in a ...,Managing sepsis in a critical care unit involv...,2,5
1,"What are the common symptoms for appendicitis,...",Common symptoms of appendicitis include:\n\n1....,4,5
2,What are the effective treatments or solutions...,"Sudden patchy hair loss, often referred to as ...",4,5
3,What treatments are recommended for a person w...,The treatment for a person who has sustained a...,3,5


#### **Evaluation 2: Prompt Engineering Response Evaluation**

In [ ]:
import re
# Question 1
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[0], response=result_df.responses_with_prompt_eng[0], max_tokens=512)

score_match = re.search(r'{groundedness_score:(\d+)}', ground)
llm_judge_prompt_ground_1 = int(score_match.group(1)) if score_match else 0

score_match = re.search(r'{relevance_score:(\d+)}', rel)
llm_judge_prompt_rel_1 = int(score_match.group(1)) if score_match else 0

print(ground,end="\n\n")
print(rel)

1. Steps to evaluate the answer as per the metric:
- The answer outlines a management protocol for sepsis in a critical care unit, including initial assessment, key tests, initial management, and escalation.
- Each claim made in the answer needs to be compared against the provided context to check for support or contradiction.

2. Step-by-step explanation:
- **Initial Assessment:** The answer mentions evaluating vital signs and signs of organ dysfunction. While the context discusses monitoring vital signs, it does not explicitly mention the assessment of organ dysfunction, which could be seen as an inference.
- **Key Tests:** The answer lists obtaining blood cultures and other tests. The context supports the need for blood cultures and mentions the importance of laboratory tests, but it does not specify the complete blood count or liver function tests as part of the protocol.
- **Initial Management:** The answer discusses fluid resuscitation, antibiotic therapy, and vasopressors. The c

In [ ]:
import re
# Question 2
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[1], response=result_df.responses_with_prompt_eng[1], max_tokens=512)  #Complete the code to calculate the groundedness and relevance score

score_match = re.search(r'{groundedness_score:(\d+)}', ground)
llm_judge_prompt_ground_2 = int(score_match.group(1)) if score_match else 0

score_match = re.search(r'{relevance_score:(\d+)}', rel)
llm_judge_prompt_rel_2 = int(score_match.group(1)) if score_match else 0

print(ground,end="\n\n")
print(rel)

1. Steps to evaluate the answer as per the metric:
- Identify key factual claims made in the answer:
  - Common symptoms of appendicitis include abdominal pain, nausea, vomiting, loss of appetite, fever, constipation or diarrhea, and abdominal swelling.
  - Appendicitis is primarily a surgical condition, and while antibiotics may be used, they are not a definitive cure.
  - The standard surgical procedure for appendicitis is an appendectomy, which can be performed as either an open or laparoscopic procedure.
  - Early diagnosis and treatment are crucial to prevent complications.
  - Management approaches may vary based on individual patient factors and local guidelines.

- Compare each claim against the provided context:
  - The symptoms listed in the answer are supported by the context, which describes abdominal pain, nausea, vomiting, loss of appetite, and fever as common symptoms.
  - The statement that appendicitis is primarily a surgical condition and that antibiotics are not a de

In [ ]:
import re
# Question 3
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[2], response=result_df.responses_with_prompt_eng[2], max_tokens=512)  #Complete the code to calculate the groundedness and relevance score

score_match = re.search(r'{groundedness_score:(\d+)}', ground)
llm_judge_prompt_ground_3 = int(score_match.group(1)) if score_match else 0

score_match = re.search(r'{relevance_score:(\d+)}', rel)
llm_judge_prompt_rel_3 = int(score_match.group(1)) if score_match else 0

print(ground,end="\n\n")
print(rel)

1. Steps to evaluate the answer as per the metric:
- Identify key factual claims made in the answer.
- Compare each claim against the provided context.
- Determine whether any part of the answer includes unsupported or external information.
- Verify that no inference beyond the context is introduced.

2. Step-by-step explanation:
- The answer identifies **Alopecia Areata** as a possible cause of sudden patchy hair loss, which is supported by the context that describes it as an autoimmune condition leading to sudden hair loss.
- It mentions **stress** as a potential trigger for hair loss, which is not explicitly stated in the context but is a common understanding in medical literature regarding hair loss.
- The claim about **infections**, specifically **tinea capitis**, causing localized hair loss is supported by the context, which mentions that treatment for tinea capitis involves antifungals.
- The mention of **nutritional deficiencies** (iron, zinc, biotin) is supported by the contex

In [ ]:
import re
# Question 4
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[3], response=result_df.responses_with_prompt_eng[3], max_tokens=512)  #Complete the code to calculate the groundedness and relevance score

score_match = re.search(r'{groundedness_score:(\d+)}', ground)
llm_judge_prompt_ground_4 = int(score_match.group(1)) if score_match else 0

score_match = re.search(r'{relevance_score:(\d+)}', rel)
llm_judge_prompt_rel_4 = int(score_match.group(1)) if score_match else 0

print(ground,end="\n\n")
print(rel)

1. Steps to evaluate the answer as per the metric:
- The answer presents several key claims regarding the treatment approaches for brain injury, including initial assessment, acute management, symptomatic treatment, rehabilitation, and long-term management.
- Each claim must be compared against the provided context to determine if they are supported by the information in the context.
- The answer should not introduce any unsupported information or reasoning that is not explicitly stated in the context.

2. Step-by-step explanation:
- **Initial Assessment and Stabilization:** The context mentions the importance of a neurological examination and imaging studies (CT or MRI) to assess the extent of injury, which supports this claim.
- **Acute Management:** The context discusses ensuring airway, breathing, and circulation, as well as managing intracranial pressure, which aligns with the claims made in the answer.
- **Symptomatic Treatment:** The context does not explicitly mention pain mana

In [ ]:
# Create a DataFrame to store the prompt engineering evaluation results
prompt_eng_evaluation_df = pd.DataFrame({
    "question": [question_1, question_2, question_3, question_4],
    "prompt_engineered_response": [
        result_df.responses_with_prompt_eng[0],
        result_df.responses_with_prompt_eng[1],
        result_df.responses_with_prompt_eng[2],
        result_df.responses_with_prompt_eng[3]
    ],
    "groundedness_score": [
        llm_judge_prompt_ground_1,
        llm_judge_prompt_ground_2,
        llm_judge_prompt_ground_3,
        llm_judge_prompt_ground_4
    ],
    "relevance_score": [
        llm_judge_prompt_rel_1,
        llm_judge_prompt_rel_2,
        llm_judge_prompt_rel_3,
        llm_judge_prompt_rel_4
    ]
})

prompt_eng_evaluation_df['groundedness_score'] = pd.to_numeric(
    prompt_eng_evaluation_df['groundedness_score'], errors='coerce'
)

prompt_eng_evaluation_df['relevance_score'] = pd.to_numeric(
    prompt_eng_evaluation_df['relevance_score'], errors='coerce'
)

display(prompt_eng_evaluation_df)


,question,prompt_engineered_response,groundedness_score,relevance_score
0,What is the protocol for managing sepsis in a ...,Answer:\n**Management Protocol for Sepsis in a...,3,5
1,"What are the common symptoms for appendicitis,...",Answer:\n\n**Common Symptoms of Appendicitis:*...,0,5
2,What are the effective treatments or solutions...,Answer:\n\n### Possible Causes:\n- **Alopecia ...,0,5
3,What treatments are recommended for a person w...,Answer:\n**Treatment Approaches for Brain Inju...,3,5


#### **Evaluation 3: RAG Response Evaluation**

In [ ]:
# Question 1
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[0], response=result_df.responses_with_RAG[0], max_tokens=512)  #Complete the code to calculate the groundedness and relevance
print(ground,end="\n\n")
print(rel)

1. Steps to evaluate the answer as per the metric:
- Identify key factual claims made in the answer.
- Compare each claim against the provided context.
- Determine whether any part of the answer includes unsupported or external information.
- Verify that no inference beyond the context is introduced.

2. Step-by-step explanation:
- The answer begins with the initial management of sepsis, stating that specimens should be obtained for Gram stain and culture before starting antibiotics. This is supported by the context.
- It mentions the need for prompt empiric therapy after suspecting sepsis, which is also directly supported by the context.
- The answer details antibiotic selection based on the suspected source, clinical setting, and previous culture results, all of which are mentioned in the context.
- The specific antibiotic regimens provided (gentamicin or tobramycin plus a 3rd-generation cephalosporin, etc.) are directly taken from the context.
- The mention of adding vancomycin if r

In [ ]:
# Question 2
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[1], response=result_df.responses_with_RAG[1], max_tokens=512)  #Complete the code to calculate the groundedness and relevance
print(ground,end="\n\n")
print(rel)

1. Steps to evaluate the answer as per the metric:
- The answer lists common symptoms of appendicitis, including epigastric or periumbilical pain, nausea, vomiting, and anorexia, which are all supported by the context.
- It mentions that appendicitis cannot be cured via medicine alone and requires surgical treatment, which is also supported by the context.
- The answer specifies that the recommended surgical procedure is either an open or laparoscopic appendectomy, which is directly supported by the context.

2. Step-by-step explanation:
- The claim about common symptoms is well-supported by the context, which details the classic symptoms of appendicitis.
- The assertion that appendicitis cannot be cured via medicine alone is supported by the context, which states that treatment is surgical removal.
- The mention of the surgical procedures (open or laparoscopic appendectomy) is also directly supported by the context, which discusses these methods as the standard treatment for appendici

In [ ]:
# Question 3
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[2], response=result_df.responses_with_RAG[2], max_tokens=512)  #Complete the code to calculate the groundedness and relevance
print(ground,end="\n\n")
print(rel)

1. Steps to evaluate the answer as per the metric:
- The answer claims that treatment options for alopecia areata are not explicitly detailed in the context, which is accurate. It also mentions that microscopic hair examination or scalp biopsy may be indicated, which is supported by the context.
- The answer states that alopecia areata is characterized by sudden patchy hair loss in individuals without obvious skin or systemic disorders, which is directly supported by the context.
- The claim that alopecia areata is thought to be an autoimmune disorder affecting genetically susceptible individuals is also supported by the context.
- The mention of environmental triggers is consistent with the context, which states that alopecia areata may be influenced by unclear environmental triggers.

2. Step-by-step explanation:
- The first claim about treatment options not being explicitly detailed is supported by the context, as it does not provide specific treatments for alopecia areata.
- The se

In [ ]:
# Question 4
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[3], response=result_df.responses_with_RAG[3], max_tokens=512)  #Complete the code to calculate the groundedness and relevance
print(ground,end="\n\n")
print(rel)

1. Steps to evaluate the answer as per the metric:
- Identify key factual claims made in the answer:
  - Initial treatment for TBI includes ensuring a reliable airway, maintaining ventilation, oxygenation, and blood pressure, and possible surgical intervention.
  - Rehabilitation is needed when neurologic deficits persist, involving a team approach with various therapies and counseling.
  - Prolonged rehabilitation is required for patients in a coma exceeding 24 hours.
  - Early intervention by rehabilitation specialists is crucial for recovery, including prevention of secondary disabilities and family education.
  - Extensive cognitive therapy may be necessary for patients with severe cognitive dysfunction.

- Compare each claim against the provided context:
  - The context supports the claims about initial treatment for TBI, including airway management, ventilation, and potential surgery.
  - The need for rehabilitation and the team approach is explicitly mentioned in the context.
  

In [ ]:
import re

for i in range(4):
    ground, rel = generate_ground_relevance_response(
        user_input=result_df.questions[i],
        response=result_df.responses_with_RAG[i],
        max_tokens=512
    )

    g = re.search(r'{groundedness_score:(\d+)}', ground)
    r = re.search(r'{relevance_score:(\d+)}', rel)

    globals()[f"llm_judge_rag_ground_{i+1}"] = int(g.group(1)) if g else 0
    globals()[f"llm_judge_rag_rel_{i+1}"] = int(r.group(1)) if r else 0


In [ ]:
# Create a DataFrame to store the RAG evaluation results
rag_evaluation_df = pd.DataFrame({
    "question": [question_1, question_2, question_3, question_4],
    "rag_response": [
        result_df.responses_with_RAG[0],
        result_df.responses_with_RAG[1],
        result_df.responses_with_RAG[2],
        result_df.responses_with_RAG[3]
    ],
    "groundedness_score": [
        llm_judge_rag_ground_1,
        llm_judge_rag_ground_2,
        llm_judge_rag_ground_3,
        llm_judge_rag_ground_4
    ],
    "relevance_score": [
        llm_judge_rag_rel_1,
        llm_judge_rag_rel_2,
        llm_judge_rag_rel_3,
        llm_judge_rag_rel_4
    ]
})

rag_evaluation_df['groundedness_score'] = pd.to_numeric(
    rag_evaluation_df['groundedness_score'], errors='coerce'
)

rag_evaluation_df['relevance_score'] = pd.to_numeric(
    rag_evaluation_df['relevance_score'], errors='coerce'
)

display(rag_evaluation_df)


,question,rag_response,groundedness_score,relevance_score
0,What is the protocol for managing sepsis in a ...,Answer:\n- **Initial Management:**\n - Obtain...,4,5
1,"What are the common symptoms for appendicitis,...",Answer:\nCommon symptoms of appendicitis inclu...,5,5
2,What are the effective treatments or solutions...,Answer:\n- **Effective Treatments for Sudden P...,5,2
3,What treatments are recommended for a person w...,Answer:\n- Initial treatment for traumatic bra...,5,5


## Actionable Insights and Business Recommendations


*   
*  
*



<font size=6 color='#4682B4'>Power Ahead</font>
___